In [1]:
!nvidia-smi

Wed Sep 30 19:39:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!sudo apt-get update -y
!sudo apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [114 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [3,044 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]      
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,25

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama started")


Ollama started


In [5]:
!curl http://127.0.0.1:11434

Ollama is running

In [6]:
!ollama run qwen3.8:latest "Dime un chiste corto de programadores"

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling ac3714bfddde:   5% ▕                  ▏  50 MB/931 MB                  pulling manifest 
pulling ac3714bfddde:  13% ▕██                ▏ 119 MB/931 MB                  pulling manifest 
pulling ac3714bfddde:  17% ▕███               ▏ 162 MB/931 MB                  pulling manifest 
pulling ac3714bfddde:  27% ▕████              ▏ 247 MB/931 MB                  pulling manifest 
pulling ac3714bfddde:  34% ▕██████            ▏ 317 MB/931 MB                  pulling manifest 
pulling ac3714bfddde:  38% ▕██████            ▏ 358 MB/931 MB                  pulling manifest 
pulling ac3714bfddde:  47% ▕████████          ▏ 434 MB/931 MB                  pulling manifest 
pulling ac3714bfddde:  55% ▕█████████         ▏ 515 MB/931 MB                  pulling manifest 
pulling ac3714bfddde:  60% ▕██████████        ▏ 559 MB/931 MB                  pulli

In [7]:
!ollama list

]11;?\NAME              ID              SIZE     MODIFIED       
qwen3.8:latest    aaee06c39dcf    17 GB    48 minutes ago    


In [8]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 125207 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
Processing triggers for man-db (2.10.2-1) ...


In [9]:
!cloudflared --version

cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)


In [14]:
import subprocess
import time

cloudflared = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url", "http://127.0.0.1:11434",
        "--http-host-header", "localhost:11434"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(8)

for _ in range(30):
    line = cloudflared.stdout.readline()
    if line:
        print(line, end="")


2026-09-30T20:37:05Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-30T20:37:05Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-30T20:37:09Z INF +--------------------------------------------------------------------------------------------+
2026-09-30T20:37:09Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-30T20:37:09Z INF |  https://calendar-exercise-epinions-affected.trycloudf

In [15]:
import requests

PUBLIC_URL = "https://calendar-exercise-epinions-affected.trycloudflare.com"

response = requests.get(PUBLIC_URL + "/api/tags", timeout=60)

print("Status:", response.status_code)
print(response.text)


Status: 200
{"models":[{"name":"qwen3.8:latest","model":"qwen3.8:latest","modified_at":"2026-09-30T19:44:59.325846983Z","size":17741872154,"digest":"aaee06c39dcf2437cde036998d960e1fc1494b8191be7cc9657d01e509097813","details":{"parent_model":"qwen3.8:27b-q4_K_M","format":"gguf","family":"qwen35","families":["qwen35"],"parameter_size":"27.3B","quantization_level":"Q4_K_M","context_length":262144,"embedding_length":5120},"capabilities":["completion","vision","tools","thinking"]}]}
